# OpenAI Word Embeddings, Semantic Search

Word embeddings are a way of representing words and phrases as vectors. They can be used for a variety of tasks, including semantic search, anomaly detection, and classification. This notebook will illustrate how words whose vectors are numerically similar are also similar in semantic meaning and we will learn how to implement semantic search using OpenAI embeddings.

In [6]:
# Import Python libraries
import os
from Utilities.envVars import *

# Read Data File Containing Words

Now that we have configured OpenAI, let's start with a simple CSV file with familiar words. From here we'll build up to a more complex semantic search using sentences from the Fed speech. [Save the linked "words.csv" as a CSV](https://gist.github.com/hackingthemarkets/25240a55e463822d221539e79d91a8d0) and upload it to Google Colab. Once the file is uploaded, let's read it into a pandas dataframe using the code below:

In [7]:
import pandas as pd
import numpy as np
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from Utilities.envVars import *

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default"
)

client = AzureOpenAI(
    azure_ad_token_provider=token_provider,
    api_version=OpenAiVersion,
    azure_endpoint=OpenAiEndPoint
)

def get_embedding(text, engine):
    return client.embeddings.create(input=[text], model=engine).data[0].embedding

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [8]:
df = pd.read_csv('./Data/CSV/words.csv')
print(df)

            text
0            red
1       potatoes
2           soda
3         cheese
4          water
5           blue
6         crispy
7      hamburger
8         coffee
9          green
10          milk
11      la croix
12        yellow
13     chocolate
14  french fries
15         latte
16          cake
17         brown
18  cheeseburger
19      espresso
20    cheesecake
21         black
22         mocha
23         fizzy
24        carbon
25        banana


# Calculate Word Embeddings

To use word embeddings for semantic search, you first compute the embeddings for a corpus of text using a word embedding algorithm. What does this mean? We are going to create a numerical representation of each of these words. To perform this computation, we'll use OpenAI's 'get_embedding' function.

Since we have our words in a pandas dataframe, we can use "apply" to apply the get_embedding function to each row in the dataframe. We then store the calculated word embeddings in a new text file called "word_embeddings.csv" so that we don't have to call OpenAI again to perform these calculations.

In [9]:
sample = get_embedding("the fox crossed the road", engine=OpenAiEmbedding)
print(f"Dimensions: {len(sample)}")
sample

Dimensions: 1536


[-0.0005743610672652721,
 0.00043588498374447227,
 -0.020444106310606003,
 0.00726999482139945,
 -0.013897964730858803,
 0.025341125205159187,
 -0.02248348295688629,
 -0.022193940356373787,
 0.013923142105340958,
 -0.03305802121758461,
 0.028979269787669182,
 0.00877434853464365,
 0.03310837596654892,
 -0.015899572521448135,
 0.0047491006553173065,
 -0.0011282654013484716,
 0.0193866528570652,
 -0.0010204743593931198,
 0.018455086275935173,
 -0.03054027259349823,
 -0.009762564674019814,
 0.029533173888921738,
 0.0021448058541864157,
 -0.023855654522776604,
 -0.0038458588533103466,
 0.006284926552325487,
 0.015307902358472347,
 -0.013797254301607609,
 -0.002736476482823491,
 -0.011556459590792656,
 0.008100851438939571,
 0.001900899107567966,
 -0.028853382915258408,
 0.000466963421786204,
 -0.013784665614366531,
 -0.017485754564404488,
 -0.0018631329294294119,
 -0.008566634729504585,
 0.011436866596341133,
 -0.027216846123337746,
 0.026965072378516197,
 0.005170823074877262,
 -0.0164534

In [10]:
df['embedding'] = df['text'].apply(lambda x: get_embedding(x, engine=OpenAiEmbedding))
df.to_csv('./Data/CSV/word_embeddings.csv')

# Semantic Search

Now that we have our word embeddings stored, let's load them into a new dataframe and use it for semantic search. Since the 'embedding' in the CSV is stored as a string, we'll use apply() and to interpret this string as Python code and convert it to a numpy array so that we can perform calculations on it.

In [11]:
df = pd.read_csv('./Data/CSV/word_embeddings.csv')
df['embedding'] = df['embedding'].apply(eval).apply(np.array)
df

,Unnamed: 0,text,embedding
0,0,red,"[9.326533472631127e-06, -0.02476814016699791, ..."
1,1,potatoes,"[0.005080870818346739, -0.031054075807332993, ..."
2,2,soda,"[0.02577228657901287, -0.007436878979206085, -..."
3,3,cheese,"[-0.0032272646203637123, -0.008705058135092258..."
4,4,water,"[0.01912183128297329, -0.012477339245378971, 0..."
5,5,blue,"[0.005474964156746864, -0.007486246060580015, ..."
6,6,crispy,"[-0.0010263145668432117, -0.00545493233948946,..."
7,7,hamburger,"[-0.013137392699718475, -0.0018266240367665887..."
8,8,coffee,"[-0.000679703603964299, -0.01955685019493103, ..."
9,9,green,"[0.01546180434525013, -0.010975971817970276, 0..."


Let's now prompt ourselves for a search term that isn't in the dataframe. We'll use word embeddings to perform a semantic search for the words that are most similar to the word we entered. I'll first try the word "hot dog". Then we'll come back and try the word "yellow".

In [12]:
search_term = input('Enter a search term: ')

Enter a search term:  hot dog


Now that we have a search term, let's calculate an embedding or vector for that search term using the OpenAI get_embedding function.

In [13]:
# semantic search
search_term_vector = get_embedding(search_term, engine=OpenAiEmbedding)

 Once we have a vector representing that word, we can see how similar it is to other words in our dataframe by calculating the cosine similarity of our search term's word vector to each word embedding in our dataframe.

In [14]:
df["similarities"] = df['embedding'].apply(lambda x: cosine_similarity(x, search_term_vector))
df

,Unnamed: 0,text,embedding,similarities
0,0,red,"[9.326533472631127e-06, -0.02476814016699791, ...",0.811867
1,1,potatoes,"[0.005080870818346739, -0.031054075807332993, ...",0.816993
2,2,soda,"[0.02577228657901287, -0.007436878979206085, -...",0.820836
3,3,cheese,"[-0.0032272646203637123, -0.008705058135092258...",0.823422
4,4,water,"[0.01912183128297329, -0.012477339245378971, 0...",0.798422
5,5,blue,"[0.005474964156746864, -0.007486246060580015, ...",0.786413
6,6,crispy,"[-0.0010263145668432117, -0.00545493233948946,...",0.820629
7,7,hamburger,"[-0.013137392699718475, -0.0018266240367665887...",0.876688
8,8,coffee,"[-0.000679703603964299, -0.01955685019493103, ...",0.799012
9,9,green,"[0.01546180434525013, -0.010975971817970276, 0...",0.784445


# Sorting By Similarity

Now that we have calculated the similarities to each term in our dataframe, we simply sort the similarity values to find the terms that are most similar to the term we searched for. Notice how the foods are most similar to "hot dog". Not only that, it puts fast food closer to hot dog. Also some colors are ranked closer to hot dog than others. Let's go back and try the word "yellow" and walk through the results.

In [15]:
df.sort_values("similarities", ascending=False).head(20)

,Unnamed: 0,text,embedding,similarities
7,7,hamburger,"[-0.013137392699718475, -0.0018266240367665887...",0.876688
18,18,cheeseburger,"[-0.018409615382552147, 0.005138860084116459, ...",0.856879
14,14,french fries,"[0.00143189518712461, -0.01648160256445408, 0....",0.838439
3,3,cheese,"[-0.0032272646203637123, -0.008705058135092258...",0.823422
2,2,soda,"[0.02577228657901287, -0.007436878979206085, -...",0.820836
6,6,crispy,"[-0.0010263145668432117, -0.00545493233948946,...",0.820629
1,1,potatoes,"[0.005080870818346739, -0.031054075807332993, ...",0.816993
13,13,chocolate,"[0.0014886129647493362, -0.013025964610278606,...",0.816503
16,16,cake,"[-0.013660978525876999, -0.016728799790143967,...",0.812063
0,0,red,"[9.326533472631127e-06, -0.02476814016699791, ...",0.811867


# Adding Words Together

What's even more interesting is that we can add word vectors together. What happens when we add the numbers for milk and espresso, then search for the word vector most similar to milk + espresso? Let's make a copy of the original dataframe and call it food_df. We'll operate on this copy. Let's try adding word together. Let's add milk + espresso and store the results in milk_espresso_vector.

In [16]:
food_df = df.copy()

milk_vector = food_df['embedding'][10]
espresso_vector = food_df['embedding'][19]

milk_espresso_vector = milk_vector + espresso_vector
milk_espresso_vector

array([-0.02178842, -0.03218406, -0.01639923, ..., -0.00437658,
        0.00074049, -0.02937349], shape=(1536,))

Now let's find the words most similar to milk + espresso. If you have never done this before, it's pretty surprising that you can add words together like this and find similar words using numbers.

In [17]:
food_df["similarities"] = food_df['embedding'].apply(lambda x: cosine_similarity(x, milk_espresso_vector))
food_df.sort_values("similarities", ascending=False)

,Unnamed: 0,text,embedding,similarities
19,19,espresso,"[-0.022676419466733932, -0.012804877012968063,...",0.960420
10,10,milk,"[0.0008879950037226081, -0.019379185512661934,...",0.960420
15,15,latte,"[-0.015830762684345245, -0.003967970609664917,...",0.922779
22,22,mocha,"[-0.01248339656740427, -0.026130499318242073, ...",0.899186
8,8,coffee,"[-0.000679703603964299, -0.01955685019493103, ...",0.894748
3,3,cheese,"[-0.0032272646203637123, -0.008705058135092258...",0.884670
13,13,chocolate,"[0.0014886129647493362, -0.013025964610278606,...",0.883071
2,2,soda,"[0.02577228657901287, -0.007436878979206085, -...",0.874072
4,4,water,"[0.01912183128297329, -0.012477339245378971, 0...",0.866066
7,7,hamburger,"[-0.013137392699718475, -0.0018266240367665887...",0.852336


# Microsoft Earnings Call Transcript

Let's tie this back to finance. I have attached some text from a recent [Microsoft earnings call here](https://gist.github.com/hackingthemarkets/1c827a7750384fcf52c84594ef216a2d). Click on "raw" and save the file as a CSV. Upload it to Google Colab as microsoft-earnings.csv. Let's use what we just learned to perform a semantic search on sentences in the Microsoft earnings call. We'll start by reading the paragraphs into a pandas dataframe.

In [13]:
earnings_df = pd.read_csv('./Data/CSV/microsoft-earnings.csv')
earnings_df

,text
0,"Thank you, Brett. To start, I want to outline ..."
1,"With that context, this quarter, the Microsoft..."
2,It helps them align their spend with demand an...
3,We are the platform of choice for customers' S...
4,Now to data and AI. With our Microsoft Intelli...
...,...
57,Other income and expense should be roughly $10...
58,"And finally, as a reminder, for Q2 cash flow, ..."
59,And FX should decrease COGS and operating expe...
60,With the high margins in our Windows OEM busin...


Once we have the dataframe, we'll once again compute the embeddings for each line in our CSV file.

In [ ]:
earnings_df['embedding'] = earnings_df['text'].apply(lambda x: get_embedding(x, engine=OpenAiEmbedding))
earnings_df.to_csv('./Data/CSV/earnings-embeddings.csv')

If you download the earnings_embeddings.csv file locally and open it up, you'll see that our embeddings are for entire paragraphs - not just words. This means that we'll be able to search on similar sentences even if there isn't an exact match for the string we search for. We are searching on meaning.

In [ ]:
earnings_search = input("Search earnings for a sentence:")

In [ ]:
earnings_search_vector = get_embedding(earnings_search, engine=OpenAiEmbedding)

In [ ]:
earnings_df["similarities"] = earnings_df['embedding'].apply(lambda x: cosine_similarity(x, earnings_search_vector))
earnings_df

In [ ]:
earnings_df.sort_values("similarities", ascending=False)